# MilletSaarthi — All 4 Agents (Starter Code)

One notebook with scaffolded code for Agents 1–4. Each agent is self-contained: imports, class definition, and a standalone test using mock JSON. Tomorrow we will extract these into `agents/*.py` and wire them through `orchestrator.py`.

**Contract:** every agent implements `predict(state: dict) -> dict`. The orchestrator merges outputs into one accumulating state.

## 0. Shared setup

In [ ]:
import os, json, math, random, logging
from datetime import datetime, timedelta
from pathlib import Path
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

GRADE_MULTIPLIER = {'A': 1.00, 'B': 0.93, 'C': 0.85}
TRANSPORT_RATE   = 20   # INR per km per quintal
ROAD_FACTOR      = 1.3  # straight-line -> road

MOCK_FARMER_INPUT = {
    'image_path': 'sample.jpg',
    'location': {'lat': 19.8762, 'lng': 75.3433},   # Aurangabad
    'quantity_quintal': 10,
    'moisture': 12.5
}

---
# AGENT 1 — Quality Assessment (MobileNetV3)

Owner: Prem. Loads a trained checkpoint and predicts millet + grade from an image. Training lives in `train_classifier.ipynb`; this cell is the **inference agent**.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

class QualityAgent:
    def __init__(self, model_path=None, classes=None, device=None):
        self.log = logging.getLogger('QualityAgent')
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.classes = classes or [
            'Bajra grade a','Bajra grade b',
            'jowar grade a','Jowar grade b','jowar grade c',
            'Ragi grade A','Ragi grade b'
        ]
        self.model = models.mobilenet_v3_large(weights=None)
        self.model.classifier[3] = nn.Linear(self.model.classifier[3].in_features, len(self.classes))
        if model_path and Path(model_path).exists():
            self.model.load_state_dict(torch.load(model_path, map_location=self.device))
            self.log.info(f'Loaded checkpoint {model_path}')
        else:
            self.log.warning('No checkpoint — using random weights (demo mode)')
        self.model.eval().to(self.device)
        self.tf = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])

    @staticmethod
    def _parse_class(name: str):
        parts = name.lower().split()
        millet = parts[0].capitalize()
        grade  = parts[-1].upper()
        return millet, grade

    def predict(self, state: dict) -> dict:
        path = state['image_path']
        if not Path(path).exists():
            self.log.warning(f'Image not found ({path}) — returning mock output')
            return {'millet':'Bajra','grade':'A','quality_score':0.87,'confidence':0.92}
        img = self.tf(Image.open(path).convert('RGB')).unsqueeze(0).to(self.device)
        with torch.no_grad():
            probs = torch.softmax(self.model(img), 1)[0].cpu().numpy()
        idx = int(probs.argmax()); conf = float(probs[idx])
        millet, grade = self._parse_class(self.classes[idx])
        return {
            'millet': millet,
            'grade': grade,
            'quality_score': round(conf, 3),
            'confidence': round(conf, 3),
        }

# Test
q = QualityAgent()
print(json.dumps(q.predict(MOCK_FARMER_INPUT), indent=2))

---
# AGENT 2 — Price Prediction (XGBoost + LSTM ensemble)

Owner: Purva. Needs cleaned Agmarknet data in `data/agmarknet/clean.csv`. Code below scaffolds cleaning, feature engineering, XGBoost training, LSTM training, ensemble, and the `PriceAgent` inference class.

In [ ]:
import pandas as pd, numpy as np, joblib

# ---------- 2.1 Cleaning ----------
def clean_agmarknet(raw_csv, out_csv):
    df = pd.read_csv(raw_csv)
    df = df.dropna(subset=['modal_price'])
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['market','millet','date']).reset_index(drop=True)
    # winsorize
    for m, g in df.groupby('millet'):
        lo, hi = g['modal_price'].quantile([0.01,0.99])
        df.loc[g.index,'modal_price'] = g['modal_price'].clip(lo,hi)
    df.to_csv(out_csv, index=False)
    return df

# ---------- 2.2 Feature engineering ----------
def build_features(df):
    df = df.copy()
    df['month']       = df['date'].dt.month
    df['doy']         = df['date'].dt.dayofyear
    df['woy']         = df['date'].dt.isocalendar().week.astype(int)
    df['season']      = df['month'].map(lambda m: 'kharif' if 6<=m<=10 else 'rabi' if m in [11,12,1,2,3] else 'summer')
    df = pd.get_dummies(df, columns=['season','millet','market'], drop_first=False)
    # lags per market/millet would need groupby — simplified global lag here
    df['lag_7']   = df['modal_price'].shift(7)
    df['lag_30']  = df['modal_price'].shift(30)
    df['roll_30'] = df['modal_price'].rolling(30).mean()
    df = df.dropna().reset_index(drop=True)
    return df

# ---------- 2.3 XGBoost training ----------
def train_xgb(df_feat, target='modal_price'):
    from xgboost import XGBRegressor
    from sklearn.metrics import mean_absolute_percentage_error
    df = df_feat.sort_values('date')
    cut = df['date'].quantile(0.85)
    train = df[df['date']<=cut]; val = df[df['date']>cut]
    feat_cols = [c for c in df.columns if c not in ['date',target]]
    m = XGBRegressor(n_estimators=500, max_depth=6, learning_rate=0.05,
                     early_stopping_rounds=30, eval_metric='mape')
    m.fit(train[feat_cols], train[target],
          eval_set=[(val[feat_cols], val[target])], verbose=False)
    mape = mean_absolute_percentage_error(val[target], m.predict(val[feat_cols]))
    print(f'XGB val MAPE: {mape:.4f}')
    return m, feat_cols

# ---------- 2.4 LSTM training ----------
class PriceLSTM(nn.Module):
    def __init__(self, hidden=64, layers=2):
        super().__init__()
        self.lstm = nn.LSTM(1, hidden, layers, batch_first=True)
        self.fc   = nn.Linear(hidden, 1)
    def forward(self, x):
        o,_ = self.lstm(x)
        return self.fc(o[:,-1])

def make_sequences(series, window=30):
    X,Y=[],[]
    for i in range(len(series)-window):
        X.append(series[i:i+window]); Y.append(series[i+window])
    return np.array(X, dtype=np.float32)[:,:,None], np.array(Y, dtype=np.float32)

def train_lstm(series, epochs=30, window=30):
    X,Y = make_sequences(series, window)
    n = int(len(X)*0.85)
    Xtr, Ytr, Xv, Yv = X[:n], Y[:n], X[n:], Y[n:]
    model = PriceLSTM()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    for e in range(epochs):
        model.train()
        p = model(torch.tensor(Xtr)).squeeze()
        l = loss_fn(p, torch.tensor(Ytr))
        opt.zero_grad(); l.backward(); opt.step()
        if (e+1)%5==0: print(f'LSTM epoch {e+1} loss {l.item():.2f}')
    return model

# ---------- 2.5 Agent ----------
class PriceAgent:
    def __init__(self, xgb=None, lstm=None, feat_cols=None, history=None):
        self.log = logging.getLogger('PriceAgent')
        self.xgb = xgb; self.lstm = lstm
        self.feat_cols = feat_cols or []
        self.history = history  # last 30 prices for target market

    def predict(self, state: dict) -> dict:
        millet = state.get('millet','Bajra')
        grade  = state.get('grade','A')
        # Demo fallback — when models not trained
        base = {'Bajra':2500,'Jowar':2800,'Ragi':3400}.get(millet, 2500)
        mult = GRADE_MULTIPLIER.get(grade, 0.9)
        price = base * mult * random.uniform(0.95, 1.05)
        if self.xgb is not None:
            self.log.info('TODO: build feature row from state and run self.xgb.predict()')
        trend = random.choice(['rising','stable','falling'])
        return {
            'expected_price': round(price, 2),
            'confidence': 0.81,
            'trend': trend,
            'price_range': [round(price*0.95,2), round(price*1.05,2)],
        }

# Test
p = PriceAgent()
state = {**MOCK_FARMER_INPUT, **q.predict(MOCK_FARMER_INPUT)}
print(json.dumps(p.predict(state), indent=2))

---
# AGENT 3 — Market Comparison

Owner: Adhishree. No ML — scrapes APMC live prices, computes transport + commission, ranks markets by net profit.

In [ ]:
import requests
from bs4 import BeautifulSoup
try:
    from geopy.distance import geodesic
except ImportError:
    geodesic = None

# Minimal seed list — expand to 20+ in data/apmc_markets.json
APMC_MARKETS = [
    {'name':'Aurangabad APMC','district':'Aurangabad','lat':19.8762,'lng':75.3433,'commission_pct':2.0,'supported':['Bajra','Jowar','Ragi']},
    {'name':'Jalna APMC',     'district':'Jalna',     'lat':19.8347,'lng':75.8816,'commission_pct':1.8,'supported':['Bajra','Jowar']},
    {'name':'Nashik APMC',    'district':'Nashik',    'lat':20.0110,'lng':73.7903,'commission_pct':2.2,'supported':['Bajra','Jowar','Ragi']},
    {'name':'Pune APMC',      'district':'Pune',      'lat':18.5204,'lng':73.8567,'commission_pct':2.0,'supported':['Jowar','Ragi']},
    {'name':'Solapur APMC',   'district':'Solapur',   'lat':17.6599,'lng':75.9064,'commission_pct':1.5,'supported':['Bajra','Jowar']},
]

def distance_km(a_lat, a_lng, b_lat, b_lng):
    if geodesic is None:
        # haversine fallback
        R = 6371
        dlat = math.radians(b_lat - a_lat); dlng = math.radians(b_lng - a_lng)
        h = math.sin(dlat/2)**2 + math.cos(math.radians(a_lat))*math.cos(math.radians(b_lat))*math.sin(dlng/2)**2
        d = 2*R*math.asin(math.sqrt(h))
    else:
        d = geodesic((a_lat,a_lng),(b_lat,b_lng)).km
    return round(d * ROAD_FACTOR, 1)

def scrape_live_price(market_name, millet):
    """TODO: real scraper for agmarknet.gov.in. Return None on failure."""
    return None

class MarketAgent:
    def __init__(self, markets=None):
        self.log = logging.getLogger('MarketAgent')
        self.markets = markets or APMC_MARKETS

    def predict(self, state: dict) -> dict:
        millet   = state.get('millet','Bajra')
        qty      = state.get('quantity_quintal',1)
        exp_price= state.get('expected_price',2500)
        f_lat    = state['location']['lat']; f_lng = state['location']['lng']
        options = []
        for m in self.markets:
            if millet not in m['supported']: continue
            live = scrape_live_price(m['name'], millet)
            price = live if live is not None else exp_price
            d = distance_km(f_lat, f_lng, m['lat'], m['lng'])
            gross     = price * qty
            transport = d * TRANSPORT_RATE * qty
            commission= gross * (m['commission_pct']/100)
            net       = gross - transport - commission
            options.append({
                'name': m['name'], 'price_per_quintal': round(price,2),
                'distance_km': d, 'net_profit': round(net,2),
                'transport_cost': round(transport,2), 'commission': round(commission,2)
            })
        options.sort(key=lambda x: x['net_profit'], reverse=True)
        if not options:
            return {'best_market': None, 'net_profit': 0, 'distance_km': 0, 'alternatives': []}
        best = options[0]
        return {
            'best_market': best['name'],
            'net_profit': best['net_profit'],
            'distance_km': best['distance_km'],
            'price_at_best': best['price_per_quintal'],
            'alternatives': options[1:3]
        }

# Test
ma = MarketAgent()
state.update(p.predict(state))
print(json.dumps(ma.predict(state), indent=2))

---
# AGENT 4 — Decision Advisor

Owner: Palak. Rule engine + cross-agent verification. Pulls weather from OpenWeatherMap (optional), festival calendar, and shelf life rules. No ML required.

In [ ]:
SHELF_LIFE_RULES = [
    {'max_moisture':12,  'days':30},
    {'max_moisture':14,  'days':18},
    {'max_moisture':100, 'days':8},
]

FESTIVALS = [
    {'date':'2026-04-14','name':'Gudi Padwa','spike':10},
    {'date':'2026-08-27','name':'Ganesh Chaturthi','spike':15},
    {'date':'2026-10-20','name':'Diwali','spike':25},
    {'date':'2026-11-05','name':'Tulsi Vivah','spike':8},
]

def shelf_life_days(moisture):
    for r in SHELF_LIFE_RULES:
        if moisture <= r['max_moisture']:
            return r['days']
    return 7

def days_to_next_festival(today=None):
    today = today or datetime.now().date()
    upcoming = [(datetime.strptime(f['date'],'%Y-%m-%d').date(), f) for f in FESTIVALS]
    upcoming = [(d,f) for d,f in upcoming if d >= today]
    if not upcoming: return None, None
    d,f = min(upcoming, key=lambda x:x[0])
    return (d-today).days, f

def fetch_weather(lat, lng, api_key=None):
    """TODO: real OpenWeatherMap One Call. Mocked for now."""
    return {
        'rain_prob_3d': random.uniform(0,1),
        'rain_prob_7d': random.uniform(0,1),
        'max_temp': 32,
        'summary': 'Clear skies next 3 days'
    }

class DecisionAgent:
    def __init__(self, ow_key=None):
        self.log = logging.getLogger('DecisionAgent')
        self.ow_key = ow_key

    def _rules(self, state, weather, fest_days, shelf_days):
        moisture = state.get('moisture',12)
        grade    = state.get('grade','A')
        trend    = state.get('trend','stable')
        net      = state.get('net_profit',0)

        if moisture > 14 or weather['rain_prob_3d'] > 0.7:
            return 'URGENT_SELL', 'High moisture or heavy rain forecast in next 3 days'
        if trend == 'rising' and fest_days is not None and fest_days <= 7 and net > 0:
            return 'SELL_NOW', 'Price rising and festival demand within a week'
        if trend == 'rising' and weather['rain_prob_7d'] < 0.3 and shelf_days >= 15:
            return 'WAIT', 'Prices rising, weather clear, sufficient shelf life'
        if grade == 'A' and net > 0:
            return 'SELL_NOW', 'Grade A produce with positive net profit'
        if grade == 'C':
            return 'HOLD_CAUTION', 'Grade C — wait for better market or regrade'
        return 'SELL_NOW', 'Default — sell at best market'

    def _verify(self, state):
        issues = []
        if state.get('grade')=='A' and state.get('expected_price',9999) < 1800:
            issues.append('Grade A but predicted price unusually low')
        gross = state.get('expected_price',0) * state.get('quantity_quintal',1)
        if gross and state.get('net_profit',0) < 0.5*gross:
            issues.append('Transport/commission eats >50% of gross')
        if state.get('confidence',1) < 0.5:
            return 'SUSPICIOUS','HIGH', issues
        if issues:
            return 'SUSPICIOUS','MEDIUM', issues
        return 'VERIFIED','LOW', issues

    def predict(self, state: dict) -> dict:
        weather    = fetch_weather(state['location']['lat'], state['location']['lng'], self.ow_key)
        fest_days, fest = days_to_next_festival()
        shelf_days = shelf_life_days(state.get('moisture',12))
        action, reason = self._rules(state, weather, fest_days, shelf_days)
        status, risk, issues = self._verify(state)
        return {
            'action': action,
            'market': state.get('best_market'),
            'reason': reason,
            'status': status,
            'risk': risk,
            'issues': issues,
            'weather_summary': weather['summary'],
            'shelf_life_days': shelf_days,
            'next_festival': fest['name'] if fest else None,
            'days_to_festival': fest_days,
        }

# Test
d = DecisionAgent()
state.update(ma.predict(state))
print(json.dumps(d.predict(state), indent=2, default=str))

---
## End-to-end dry run with mocks

Verifies the contract flows cleanly through all 4 agents.

In [ ]:
state = dict(MOCK_FARMER_INPUT)
state.update(q.predict(state))
state.update(p.predict(state))
state.update(ma.predict(state))
state.update(d.predict(state))
print(json.dumps(state, indent=2, default=str))